# Librerias 

In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Preprocesamiento 

In [36]:
features = ["height", "weight", "base_experience", "type_main"]
target = "strong"

In [37]:
def make_preprocessor():
    numeric = Pipeline([
        ("scaler", StandardScaler())
    ])
    categorical = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric, ["height", "weight", "base_experience"]),
        ("cat", categorical, ["type_main"])
    ])
    return preprocessor

# Modelos 

In [38]:
def build_models(pre):
    models = {
        "logistic": Pipeline([
            ("pre", pre),
            ("clf", LogisticRegression(max_iter=300))
        ]),
        "svm_rbf": Pipeline([
            ("pre", pre),
            ("clf", SVC(kernel="rbf", probability=True))
        ]),
        "svm_linear": Pipeline([
            ("pre", pre),
            ("clf", SVC(kernel="linear", probability=True))
        ]),
        "bayes": Pipeline([
            ("pre", pre),
            ("clf", GaussianNB())
        ]),
        "knn": Pipeline([
            ("pre", pre),
            ("clf", KNeighborsClassifier(n_neighbors=7))
        ])
    }
    return models

In [39]:
def train_models(models, X, y):
    for name, model in models.items():
        model.fit(X, y)
    return models

In [40]:
# Prediccion de probabilidades
def predict_probas(models, X):
    probas = {}
    for name, model in models.items():
        if hasattr(model.named_steps["clf"], "predict_proba"):
            probas[name] = model.predict_proba(X)[:, 1]
        else:
            dec = model.decision_function(X)
            probas[name] = (dec - dec.min()) / (dec.max() - dec.min())
    return probas

# Metodos de Votacion 

In [41]:
def hard_vote(label_dict):
    """Votación dura (mayoría simple)"""
    labels = np.vstack(list(label_dict.values()))
    return (labels.sum(axis=0) > labels.shape[0] / 2).astype(int)


def soft_vote(proba_dict, thr=0.5):
    """Votación suave (promedio de probabilidades)"""
    probs = np.vstack(list(proba_dict.values()))
    return (probs.mean(axis=0) >= thr).astype(int)


def weighted_vote(proba_dict, weights, thr=0.5):
    """Votación ponderada"""
    keys = list(proba_dict.keys())
    W = np.array([weights[k] for k in keys])
    P = np.vstack([proba_dict[k] for k in keys])
    score = np.average(P, axis=0, weights=W)
    return (score >= thr).astype(int)

In [42]:
# Pesos segun desempeño promedio en validacion cruzada
def cv_weights(models, X, y, metric="f1", cv=5):
    scores = {}
    scoring = metric if metric in ["accuracy", "f1"] else "accuracy"
    for name, model in models.items():
        s = cross_val_score(model, X, y, scoring=scoring, cv=cv).mean()
        scores[name] = max(s, 1e-6)
    total = sum(scores.values())
    return {k: v / total for k, v in scores.items()}

In [43]:
def train_and_predict(train_csv, test_csv, output_csv=None):
    """
    Entrena modelos con el dataset de entrenamiento
    y predice sobre el dataset de prueba.
    Devuelve métricas de accuracy y F1 .
    """
    df_train = pd.read_csv(train_csv)
    df_test = pd.read_csv(test_csv)

    X_train = df_train[features]
    y_train = df_train[target].astype(int)

    X_test = df_test[features]
    y_test = df_test[target].astype(int) if target in df_test.columns else None

    pre = make_preprocessor()
    models = build_models(pre)
    train_models(models, X_train, y_train)


    probas_test = predict_probas(models, X_test)
    labels_test = {k: (v >= 0.5).astype(int) for k, v in probas_test.items()}

    # Votaciones
    y_hard = hard_vote(labels_test)
    y_soft = soft_vote(probas_test)
    weights = cv_weights(models, X_train, y_train)
    y_weighted = weighted_vote(probas_test, weights)

    # Guardar predicciones 
    df_test["pred_hard"] = y_hard
    df_test["pred_soft"] = y_soft
    df_test["pred_weighted"] = y_weighted

    if output_csv:
        df_test.to_csv(output_csv, index=False)
        print(f" Predicciones guardadas en: {output_csv}")

    # Evaluar si tiene etiquetas reales
    if y_test is not None:
        acc_hard = accuracy_score(y_test, y_hard)
        acc_soft = accuracy_score(y_test, y_soft)
        acc_weighted = accuracy_score(y_test, y_weighted)
        f1_hard = f1_score(y_test, y_hard)
        f1_soft = f1_score(y_test, y_soft)
        f1_weighted = f1_score(y_test, y_weighted)

        metrics = pd.DataFrame({
            "Modelo": ["Votación Dura", "Votación Suave", "Votación Ponderada"],
            "Accuracy": [acc_hard, acc_soft, acc_weighted],
            "F1 Score": [f1_hard, f1_soft, f1_weighted]
        })
        print("Desempeño de los ensambles:\n")
        print(metrics.to_string(index=False))
        return metrics
    else:
        print(" El CSV de prueba no tiene la columna 'strong'. Solo se generaron predicciones.")
        return None

# Entrenamiento y prueba 

In [44]:
train_csv = r"D:\Ciencia de Datos\7mo\Aprendizaje_automatico\Practicas\7\pokemon_data_sample_balanced.csv"
test_csv  = r"D:\Ciencia de Datos\7mo\Aprendizaje_automatico\Practicas\7\testset.csv"
output_csv = r"D:\Ciencia de Datos\7mo\Aprendizaje_automatico\Practicas\7\pokemon_predictions.csv"

metrics = train_and_predict(train_csv, test_csv, output_csv)


 Predicciones guardadas en: D:\Ciencia de Datos\7mo\Aprendizaje_automatico\Practicas\7\pokemon_predictions.csv
Desempeño de los ensambles:

            Modelo  Accuracy  F1 Score
     Votación Dura  0.891667  0.811594
    Votación Suave  0.916667  0.857143
Votación Ponderada  0.900000  0.828571
